## 02 — Transaction Generation (Pareto skew + injected fraud signal)
 Run after 01_config_and_dimensions. Reads customer_ids / merchant_ids from
 the scratch schema to guarantee referential integrity, then generates
 750k-900k transactions across a 12-18 month window with:
   - 80/20 Pareto skew (20% of customers drive ~80% of transaction volume)
   - ~1.5-2% fraud-flagged transactions with distinguishable patterns
     (odd-hour timestamps, abnormally high amounts, country mismatch vs
     customer's typical/home country)

In [ ]:
dbutils.widgets.text("catalog", "northstar_dev")
dbutils.widgets.text("schema", "raw")
dbutils.widgets.text("volume", "landing")
dbutils.widgets.text("num_transactions", "820000")
dbutils.widgets.text("months_of_history", "15")
dbutils.widgets.text("fraud_rate", "0.018")
dbutils.widgets.text("random_seed", "42")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
NUM_TXNS = int(dbutils.widgets.get("num_transactions"))
MONTHS_HISTORY = int(dbutils.widgets.get("months_of_history"))
FRAUD_RATE = float(dbutils.widgets.get("fraud_rate"))
SEED = int(dbutils.widgets.get("random_seed"))

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
TXN_PATH = f"{VOLUME_ROOT}/transactions"

print(f"Target rows: {NUM_TXNS:,} | history: {MONTHS_HISTORY} months | fraud rate: {FRAUD_RATE:.2%}")


## Load reference pools (guarantees referential integrity by construction)
We never invent a customer_id/merchant_id — every transaction draws from
IDs that already exist in the dimension tables written in notebook 01.

In [ ]:
customers_ref = spark.table(f"{CATALOG}.datagen_scratch.customer_ids")
merchants_ref = spark.table(f"{CATALOG}.datagen_scratch.merchant_ids")

customers_pdf = customers_ref.toPandas()
merchants_pdf = merchants_ref.toPandas()

print(f"Customer pool: {len(customers_pdf):,} | Merchant pool: {len(merchants_pdf):,}")

## Build the Pareto customer weighting
20% of customers ("heavy" segment) receive roughly 80% of transaction mass.
Implemented as a weight vector consumed by `numpy.random.choice`, which is
the cleanest way to express Pareto skew without hand-rolling a sampler.

In [ ]:
import numpy as np
import pandas as pd

rng_master = np.random.default_rng(SEED)

n_cust = len(customers_pdf)
heavy_cutoff = int(n_cust * 0.20)

# Shuffle once so "heavy" customers aren't just the first N rows by ID order
shuffled_idx = rng_master.permutation(n_cust)
heavy_idx = set(shuffled_idx[:heavy_cutoff].tolist())

weights = np.array([16.0 if i in heavy_idx else 1.0 for i in range(n_cust)])
weights = weights / weights.sum()

customers_pdf["txn_weight"] = weights
customers_pdf["is_heavy"] = customers_pdf.index.isin(heavy_idx)

# Sanity check the skew actually lands near 80/20 before generating 800k rows
sample_draw = rng_master.choice(n_cust, size=200_000, p=weights)
heavy_share = np.isin(sample_draw, list(heavy_idx)).mean()
print(f"Sanity check — share of volume from heavy 20% of customers: {heavy_share:.1%}")

## Assign each customer a "home country" 
Needed so fraud injection can later create a genuine country-mismatch
signal (transaction country != customer's typical/home country).

In [ ]:
COUNTRY_POOL = ["US", "GB", "CA", "DE", "FR", "AU", "SG", "NL", "IE", "MX"]
COUNTRY_WEIGHTS = [0.45, 0.12, 0.10, 0.08, 0.06, 0.05, 0.04, 0.04, 0.03, 0.03]

home_countries = rng_master.choice(COUNTRY_POOL, size=n_cust, p=COUNTRY_WEIGHTS)
customers_pdf["home_country"] = home_countries

## Distributed generation
Broadcast the (small) customer/merchant reference frames to every executor,
then generate transaction rows in parallel via mapInPandas. Fraud injection
happens inline per-row so the flag and the "tell" (odd hour / high amount /
country mismatch) are coherent rather than randomly disjoint.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, BooleanType
)
from datetime import datetime, timedelta
import uuid

END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=MONTHS_HISTORY * 30)

MCC_POOL = merchants_pdf["mcc_code"].unique().tolist()
CHANNELS = ["POS", "online", "ATM"]
CHANNEL_WEIGHTS = [0.55, 0.35, 0.10]
TXN_TYPES = ["purchase", "refund", "withdrawal"]
TXN_TYPE_WEIGHTS = [0.88, 0.07, 0.05]
STATUS_BASE_WEIGHTS = {"approved": 0.93, "declined": 0.06, "flagged": 0.01}

NUM_PARTITIONS = 64
ROWS_PER_PARTITION = NUM_TXNS // NUM_PARTITIONS

partition_seed_pdf = pd.DataFrame({"partition_id": list(range(NUM_PARTITIONS))})
partition_seed_df = spark.createDataFrame(partition_seed_pdf).repartition(NUM_PARTITIONS, "partition_id")

# Broadcast reference data as plain python objects via closure — mapInPandas
# ships the function + its closed-over variables to each executor once.
customers_records = customers_pdf.to_dict("records")
merchants_records = merchants_pdf.to_dict("records")

def generate_transactions_partition(pdf_iter):
    cust_records = customers_records
    merch_records = merchants_records
    cust_weights = np.array([c["txn_weight"] for c in cust_records])
    n_cust_local = len(cust_records)
    n_merch_local = len(merch_records)

    for pdf in pdf_iter:
        part_id = int(pdf["partition_id"].iloc[0])
        local_rng = np.random.default_rng(SEED + 1000 + part_id)
        n_rows = ROWS_PER_PARTITION

        cust_draws = local_rng.choice(n_cust_local, size=n_rows, p=cust_weights)
        merch_draws = local_rng.choice(n_merch_local, size=n_rows)
        is_fraud_draws = local_rng.random(n_rows) < FRAUD_RATE

        records = []
        span_seconds = int((END_DATE - START_DATE).total_seconds())

        for i in range(n_rows):
            cust = cust_records[cust_draws[i]]
            merch = merch_records[merch_draws[i]]
            is_fraud = bool(is_fraud_draws[i])

            # --- base (non-fraud) field generation ---
            ts_offset = int(local_rng.integers(0, span_seconds))
            txn_ts = START_DATE + timedelta(seconds=ts_offset)

            # Normal spend: lognormal centered around a modest everyday amount
            amount = float(np.round(local_rng.lognormal(mean=3.2, sigma=0.9), 2))
            country = cust["home_country"]
            status = local_rng.choice(
                list(STATUS_BASE_WEIGHTS.keys()),
                p=list(STATUS_BASE_WEIGHTS.values())
            )

            if is_fraud:
                # --- coherent fraud "tell" pattern ---
                # 1) odd-hour bias: push timestamp toward 1am-4am local-ish window
                fraud_hour = int(local_rng.integers(1, 5))
                txn_day = START_DATE + timedelta(seconds=ts_offset)
                txn_ts = txn_day.replace(
                    hour=fraud_hour,
                    minute=int(local_rng.integers(0, 60)),
                    second=int(local_rng.integers(0, 60)),
                )
                # 2) abnormally high amount: 8x-25x typical spend
                amount = float(np.round(amount * local_rng.uniform(8, 25), 2))
                # 3) country mismatch vs customer's home country
                mismatch_pool = [c for c in COUNTRY_POOL if c != cust["home_country"]]
                country = mismatch_pool[int(local_rng.integers(0, len(mismatch_pool)))]
                status = "flagged"

            # Quarantine-bait: deliberately inject a small number of dirty rows
            # so Silver's DQ expectations have something real to catch.
            inject_dirty = local_rng.random() < 0.002
            null_customer_id = False
            if inject_dirty:
                dirty_choice = int(local_rng.integers(0, 2))
                if dirty_choice == 0:
                    amount = -abs(amount)  # negative amount -> DQ rule catches this
                else:
                    null_customer_id = True  # null customer_id -> DQ rule catches this

            records.append({
                "transaction_id": str(uuid.uuid4()),
                "card_id": f"CARD-{cust['customer_id'][-7:]}-{int(local_rng.integers(1,3))}",
                "customer_id": None if null_customer_id else cust["customer_id"],
                "merchant_id": merch["merchant_id"],
                "txn_timestamp": txn_ts,
                "amount": amount,
                "currency": "USD",
                "mcc_code": merch["mcc_code"],
                "channel": str(local_rng.choice(CHANNELS, p=CHANNEL_WEIGHTS)),
                "txn_type": str(local_rng.choice(TXN_TYPES, p=TXN_TYPE_WEIGHTS)),
                "status": status,
                "country": country,
                "is_fraud_flag": is_fraud,
            })

        yield pd.DataFrame.from_records(records)


txn_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("card_id", StringType(), False),
    StructField("customer_id", StringType(), True),   # nullable: dirty-row injection target
    StructField("merchant_id", StringType(), False),
    StructField("txn_timestamp", TimestampType(), False),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("mcc_code", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("txn_type", StringType(), False),
    StructField("status", StringType(), False),
    StructField("country", StringType(), False),
    StructField("is_fraud_flag", BooleanType(), False),
])

transactions_df = partition_seed_df.mapInPandas(
    generate_transactions_partition, schema=txn_schema
)
# transactions_df.cache()

actual_count = transactions_df.count()
print(f"Generated {actual_count:,} transactions")

## Validate skew & fraud rate landed where expected before writing to disk

In [ ]:
from pyspark.sql import functions as F

fraud_check = transactions_df.agg(
    F.avg(F.col("is_fraud_flag").cast("int")).alias("actual_fraud_rate")
).collect()[0]
print(f"Actual fraud rate: {fraud_check['actual_fraud_rate']:.2%} (target {FRAUD_RATE:.2%})")

null_cust_check = transactions_df.filter(F.col("customer_id").isNull()).count()
negative_amt_check = transactions_df.filter(F.col("amount") < 0).count()
print(f"Injected dirty rows — null customer_id: {null_cust_check:,} | negative amount: {negative_amt_check:,}")

date_range = transactions_df.agg(
    F.min("txn_timestamp").alias("min_ts"), F.max("txn_timestamp").alias("max_ts")
).collect()[0]
print(f"Date range: {date_range['min_ts']} -> {date_range['max_ts']}")


## Land as partitioned CSV (daily batches, simulating realistic drops)
Partitioning by date on write — not as a Delta table property, just a
folder layout — mirrors how a real card processor would drop daily files.

In [ ]:
transactions_with_date = (transactions_df
    .withColumn("txn_date", F.to_date("txn_timestamp"))
    .withColumn("txn_year", F.date_format("txn_timestamp", "yyyy"))
    .withColumn("txn_month", F.date_format("txn_timestamp", "MM")))

(transactions_with_date
    .write.mode("overwrite")
    .partitionBy("txn_year", "txn_month")
    .option("header", True)
    .csv(TXN_PATH)
)

print(f"Transactions landed at: {TXN_PATH} (partitioned by txn_year, txn_month)")

## Cleanup broadcast variables

In [ ]:
# customers_bc.unpersist()
# merchants_bc.unpersist()
print("Done. Proceed to 03_device_session_logs.py")